# Población y densidad de los cantones de Costa Rica en 2022

**Autor:** Manuel Vargas

Ejemplo de la [tarea 2](https://gf0657-programacionsig.github.io/2026-ii/tarea-02/) del curso GF-0657 Programación en SIG (II ciclo 2026). Los gráficos son de plotly: **no se ven en el visor de GitHub**. Para verlos, abra el cuaderno en Colab y ejecute todas las celdas (menú *Entorno de ejecución > Ejecutar todas*).

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii-tarea-02-ejemplo/blob/main/tarea-02.ipynb)

## Introducción

En 2022, el Instituto Nacional de Estadística y Censos (INEC) publicó estimaciones de población y vivienda para las 82 unidades de la división territorial administrativa vigente al 2011 (INEC, 2023). Este cuaderno analiza cómo se distribuye la población entre los cantones del país y responde tres preguntas:

1. ¿Cuáles son los cantones más poblados y cuánta población concentran?
2. ¿Cómo se distribuye la densidad de población entre los cantones?
3. ¿Los cantones más extensos son también los más poblados?

Los datos provienen del archivo Excel *Resultados Estimación de Población y Vivienda 2022* del INEC, del que se preparó un archivo CSV con una fila por cantón. Las variables principales son:

| Variable | Descripción | Unidad |
|---|---|---|
| `provincia`, `canton` | Nombres de la provincia y del cantón | texto |
| `poblacion` | Población estimada al 30 de junio de 2022 | habitantes |
| `area_km2` | Superficie del cantón | km² |
| `densidad` | Población por unidad de superficie | habitantes/km² |

## Carga y preparación de los datos

Se cargan las bibliotecas y se define el estilo de los gráficos: la plantilla `plotly_white` y una paleta con los colores del curso.

In [1]:
import pandas as pd
import plotly.express as px

AZUL = "#1F77B4"
NARANJA = "#D97B1B"
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = [AZUL, NARANJA]

Los datos se cargan desde la dirección *Raw* del archivo CSV en el repositorio de GitHub, no desde una ruta local, para que el cuaderno se ejecute en cualquier lugar.

In [2]:
URL = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii-tarea-02-ejemplo/main/cantones-2022.csv"
cantones = pd.read_csv(URL)

# Dimensiones del DataFrame: (filas, columnas)
cantones.shape

(82, 16)

In [3]:
# Tipos de datos de cada columna
cantones.dtypes

codigo_provincia                int64
provincia                         str
codigo_canton                   int64
canton                            str
poblacion                       int64
hombres                         int64
mujeres                         int64
viviendas                       int64
viviendas_ocupadas              int64
viviendas_desocupadas           int64
promedio_ocupantes            float64
area_km2                      float64
densidad                      float64
tasa_crecimiento_poblacion    float64
tasa_crecimiento_viviendas    float64
relacion_hombre_mujer         float64
dtype: object

In [4]:
# Primeras filas
cantones.head()

,codigo_provincia,provincia,codigo_canton,canton,poblacion,hombres,mujeres,viviendas,viviendas_ocupadas,viviendas_desocupadas,promedio_ocupantes,area_km2,densidad,tasa_crecimiento_poblacion,tasa_crecimiento_viviendas,relacion_hombre_mujer
0,1,San José,101,San José,352381,168817,183564,116872,111228,5644,3.16,44.62,7897.4,1.83,2.68,92.0
1,1,San José,102,Escazú,71500,33923,37577,26392,24379,2013,2.92,34.53,2070.7,2.14,3.14,90.3
2,1,San José,103,Desamparados,223226,106209,117017,74568,70380,4188,3.16,118.89,1877.6,0.62,1.96,90.8
3,1,San José,104,Puriscal,38525,18685,19840,14902,13438,1464,2.87,555.04,69.4,1.41,2.76,94.2
4,1,San José,105,Tarrazú,17810,9323,8487,7331,5778,1553,3.07,291.27,61.1,0.82,2.22,109.9


El archivo tiene 82 filas, una por cantón, y las columnas numéricas se leyeron con los tipos correctos. Para el análisis solo se necesitan las columnas de identificación, población, superficie y densidad, así que se seleccionan y se agrega la columna `poblacion_miles`, más cómoda para etiquetar los gráficos.

In [5]:
# Selección de columnas
cantones = cantones[["provincia", "canton", "poblacion", "area_km2", "densidad"]].copy()

# Columna nueva: población en miles de habitantes, con un decimal
cantones["poblacion_miles"] = (cantones["poblacion"] / 1000).round(1)

cantones.head()

,provincia,canton,poblacion,area_km2,densidad,poblacion_miles
0,San José,San José,352381,44.62,7897.4,352.4
1,San José,Escazú,71500,34.53,2070.7,71.5
2,San José,Desamparados,223226,118.89,1877.6,223.2
3,San José,Puriscal,38525,555.04,69.4,38.5
4,San José,Tarrazú,17810,291.27,61.1,17.8


## Tabla: población y superficie por provincia

La tabla 1 resume los datos por provincia. La densidad de cada provincia se calcula como el cociente entre su población total y su superficie total; no se promedian las densidades de los cantones, porque cantones muy distintos en tamaño pesarían igual.

In [6]:
provincias = (
    cantones.groupby("provincia")
    .agg(cantones=("canton", "count"), poblacion=("poblacion", "sum"), area_km2=("area_km2", "sum"))
    .reset_index()
)
provincias["densidad"] = provincias["poblacion"] / provincias["area_km2"]
provincias = provincias.sort_values("poblacion", ascending=False)

# Formato para presentar: separadores de miles, decimales y sin índice
provincias.style.format({
    "poblacion": "{:,.0f}",
    "area_km2": "{:,.1f}",
    "densidad": "{:,.1f}",
}).hide(axis="index").set_caption("Tabla 1. Cantones, población, superficie y densidad por provincia, 2022.")

provincia,cantones,poblacion,area_km2,densidad
San José,20,"1,601,167","4,969.8",322.2
Alajuela,16,"1,035,464","9,772.1",106.0
Cartago,8,"545,092","3,093.2",176.2
Puntarenas,11,"500,166","11,275.0",44.4
Heredia,10,"479,117","2,663.3",179.9
Limón,6,"470,383","9,176.9",51.3
Guanacaste,11,"412,808","10,196.3",40.5


San José concentra casi un tercio de la población del país en menos del 10 % de su superficie, por lo que su densidad casi duplica la de Heredia y Cartago, las siguientes. Puntarenas, Guanacaste y Limón, tres de las cuatro provincias más extensas, son las de menor densidad, con menos de 55 habitantes por km².

## Gráfico 1: los diez cantones más poblados

Para comparar una magnitud entre categorías se usa un gráfico de barras; horizontales, porque los nombres de los cantones son largos. Al pasar el cursor sobre una barra se muestra la provincia y la población exacta.

In [7]:
top10 = cantones.sort_values("poblacion", ascending=False).head(10)

fig = px.bar(
    top10,
    x="poblacion_miles",
    y="canton",
    orientation="h",
    hover_data={"provincia": True, "poblacion": ":,", "poblacion_miles": False},
    title="Figura 1. Los diez cantones más poblados de Costa Rica, 2022",
    labels={"poblacion_miles": "Población (miles de habitantes)", "canton": "Cantón", "provincia": "Provincia", "poblacion": "Población"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

San José y Alajuela, las cabeceras de las dos provincias más pobladas, encabezan la lista con más de 300 000 habitantes cada uno. Los diez cantones del gráfico, seis de ellos de la Gran Área Metropolitana, suman casi 2 millones de personas: cerca del 40 % de la población del país vive en apenas diez de los 82 cantones.

## Gráfico 2: distribución de la densidad de población

Para ver cómo se distribuye una variable numérica se usa un histograma. Cada barra cuenta cuántos cantones tienen una densidad dentro de un intervalo de 500 habitantes por km².

In [8]:
fig = px.histogram(
    cantones,
    x="densidad",
    nbins=20,
    title="Figura 2. Distribución de la densidad de población de los cantones, 2022",
    labels={"densidad": "Densidad (habitantes/km²)"},
)
fig.update_layout(yaxis_title="Cantidad de cantones", bargap=0.05)
fig.show()

La distribución es muy asimétrica: 59 de los 82 cantones tienen menos de 500 habitantes por km² (la mediana es de 83) y solo once, todos del Valle Central, superan los 2000. El valor extremo de la derecha es Tibás, con más de 9000 habitantes por km², seguido por el cantón de San José, con casi 8000. Por esa asimetría, el promedio de las densidades cantonales no representa bien al cantón típico; la mediana sí.

## Gráfico 3: superficie y población

Para explorar la relación entre dos variables numéricas se usa un gráfico de dispersión. Como ambas variables abarcan varios órdenes de magnitud, los dos ejes usan escala logarítmica. El color distingue la provincia y el cursor muestra el nombre de cada cantón.

In [9]:
fig = px.scatter(
    cantones,
    x="area_km2",
    y="poblacion",
    color="provincia",
    hover_name="canton",
    log_x=True,
    log_y=True,
    color_discrete_sequence=px.colors.qualitative.D3,
    title="Figura 3. Superficie y población de los cantones, 2022",
    labels={"area_km2": "Superficie (km², escala logarítmica)", "poblacion": "Población (habitantes, escala logarítmica)", "provincia": "Provincia"},
)
fig.show()

No hay una relación simple entre la superficie y la población. Entre los cuatro cantones más extensos, San Carlos y Pococí están entre los diez más poblados, pero Talamanca y Buenos Aires tienen menos de 51 000 habitantes. Y los cantones más poblados del Valle Central (San José, Goicoechea, Desamparados) tienen menos de 120 km². Al hacer clic en una provincia de la leyenda se oculta o se muestra: así se aprecia que los cantones de San José y Heredia se agrupan arriba a la izquierda, con poca superficie y mucha población, mientras que los de Guanacaste, Limón y Puntarenas se ubican más a la derecha y más abajo.

## Conclusiones

1. Diez cantones, seis de ellos de la Gran Área Metropolitana, concentran cerca del 40 % de la población del país. San José y Alajuela superan los 300 000 habitantes.
2. La densidad de población de los cantones es muy asimétrica: la mayoría tiene menos de 500 habitantes por km² y solo once cantones del Valle Central superan los 2000.
3. La superficie de un cantón no predice su población: entre los más extensos hay cantones muy poblados (San Carlos, Pococí) y poco poblados (Talamanca, Buenos Aires), y los más poblados del Valle Central son pequeños.

La principal limitación del análisis es la escala: el cantón es una unidad muy heterogénea (San José tiene 45 km² y San Carlos más de 3300) y dentro de cada uno la población se distribuye de forma desigual. Un análisis por distrito, con los mismos datos del INEC, mostraría patrones más finos.

## Referencias bibliográficas

Instituto Nacional de Estadística y Censos. (2023). *Resultados Estimación de Población y Vivienda 2022* [Conjunto de datos]. INEC. https://admin.inec.cr/sites/default/files/2023-11/reResultadosEstimacionPoblacionVivienda2022_3.xlsx
\
\
Plotly Technologies Inc. (s. f.). *Plotly Express in Python*. Recuperado el 29 de setiembre de 2026, de https://plotly.com/python/plotly-express/
\
\
The pandas development team. (s. f.). *pandas documentation*. Recuperado el 29 de setiembre de 2026, de https://pandas.pydata.org/docs/

## Declaración de uso de asistentes de IA

El código y el texto de este cuaderno se generaron con el asistente Claude Code (Anthropic), a partir de las indicaciones del profesor del curso, quien revisó el código, ejecutó el cuaderno y comprobó los resultados contra los datos del INEC. El asistente se usó para generar el código de los tres gráficos de plotly y el de la tabla con formato, y para redactar los párrafos de interpretación, que se verificaron con los valores de las tablas.